# Session 12: Topic Modelling with LDA

## Overview

Clustering (Part 3) puts each item into exactly one group. Text rarely works like that: a post about the cost of
a NASA mission is about **space** and about **money**; a post about sports injuries is about **sport** and about
**medicine**. **Topic modelling** treats every document as a **mixture of topics**, and every topic as a mixture of
words, and discovers both from raw text with no labels.

You will fit **Latent Dirichlet Allocation (LDA)**, the classic topic model, to a few thousand Usenet posts, read
and name the topics it finds, see how each post mixes them, choose the number of topics, and compare LDA with
**NMF**, a faster alternative. As in Session 10, the posts' real newsgroups are hidden and used only to check the
result.

By the end, you will be able to:

- Prepare a bag-of-words matrix for topic modelling, and explain why LDA needs counts.
- Explain what LDA assumes about how documents are written, and what its two outputs mean.
- Fit LDA with scikit-learn, read its topics, and interpret a document's topic mixture.
- Choose the number of topics with held-out perplexity and with human judgement, and explain why they can disagree.
- Fit NMF on TF-IDF features and compare its topics with LDA's.
- Assign topic mixtures to new documents.

**Prerequisites:** Sessions 6-8 (bag of words, TF-IDF, 20 Newsgroups) and Session 10 (clustering, ARI).

**Dataset:** [20 Newsgroups](https://archive.ics.uci.edu/dataset/113/twenty+newsgroups), as in Session 8. We use
six groups with different subjects: `comp.graphics`, `rec.autos`, `rec.sport.hockey`, `sci.med`, `sci.space` and
`talk.politics.guns`, with headers, signatures and quotes removed.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF **← you are here** |

**This session's question:** What themes run through a few thousand posts, if nobody tells us what to look for?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| scikit-learn | Dataset download, vectorisers, LDA, NMF, metrics |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score

from importlib.metadata import version

for package in ["scikit-learn", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the posts

Very short posts ("Thanks!", "Me too") carry almost no topical information and make the topics noisier, so we keep
posts with at least 20 words. The newsgroup of each post goes into `group`, hidden until Step 7.

In [ ]:
GROUPS = ["comp.graphics", "rec.autos", "rec.sport.hockey", "sci.med", "sci.space", "talk.politics.guns"]
STRIP = ("headers", "footers", "quotes")


def load(subset):
    data = fetch_20newsgroups(subset=subset, categories=GROUPS, remove=STRIP, random_state=0)
    documents = np.array(data.data, dtype=object)
    keep = np.array([len(d.split()) >= 20 for d in documents])
    return documents[keep], np.array(data.target_names)[data.target[keep]]


documents, group = load("train")
test_documents, _ = load("test")
print(f"{len(documents):,} training posts and {len(test_documents):,} held-out posts with at least 20 words")

**Read the output:** 3,152 training posts remain (359 short ones were dropped), plus 2,062 held-out posts from the
test period, which Step 8 uses to compare models.

## Step 3: A bag of words, by count

LDA models how often each word occurs, so it needs **raw counts**, not TF-IDF weights (Session 7). The vectoriser
settings shape the topics as much as the model does:

- `stop_words="english"` drops "the", "and", "of": they are in every topic and describe none.
- `max_df=0.5` drops words found in more than half of the posts, for the same reason.
- `min_df=5` drops words in fewer than 5 posts: typos and rare names that cannot define a topic.
- the token pattern keeps only words of 3 or more letters, dropping numbers and fragments.

In [ ]:
vectorizer = CountVectorizer(max_df=0.5, min_df=5, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
counts = vectorizer.fit_transform(documents)
words = vectorizer.get_feature_names_out()
print(f"document-term matrix: {counts.shape[0]:,} posts x {counts.shape[1]:,} words")
print(f"average post: {counts.sum(axis=1).mean():.0f} counted words")

**Read the output:** 3,152 posts × 7,250 words. After removing stop words and rare words, the average post keeps
80 counted words.

## Step 4: How LDA thinks documents are written

LDA imagines that each document was written like this:

1. Pick the document's **topic mixture**, for example 70% space, 20% politics, 10% computers.
2. For each word: pick a topic according to that mixture, then pick a word according to that **topic's word
   distribution** (the space topic often produces "orbit", "nasa", "launch").

Nobody writes like this, but the model is useful because we can run it **backwards**: given only the words, LDA
finds the topics (word distributions) and the mixtures that best explain them. "Dirichlet" refers to the prior
distribution over mixtures; its effect is that each document tends to use only a few topics, and each topic only a
few words strongly.

LDA has two outputs:

- `components_`: one row per topic, giving the weight of every word in it. Its top words are how we read a topic.
- `transform(counts)`: one row per document, giving its topic mixture (the shares add up to 1).

As with k-means, **we choose the number of topics**. We start with 6, the number of newsgroups.

In [ ]:
lda = LatentDirichletAllocation(n_components=6, learning_method="batch", max_iter=30, random_state=0)
mixtures = lda.fit_transform(counts)


def show_topics(model, vocabulary, n_words=10):
    for t, weights in enumerate(model.components_):
        print(f"topic {t}: {' '.join(vocabulary[np.argsort(weights)[::-1][:n_words]])}")


show_topics(lda, words)

**Read the output:** this takes about 10 seconds. Most topics can be named at a glance:

- **hockey** (team, game, hockey, season, nhl, players);
- **space** (space, nasa, launch, earth, satellite, orbit, shuttle);
- **computer graphics and files** (image, graphics, software, file, ftp; "edu" and "com" are pieces of email and
  web addresses);
- **guns and the law** (gun, guns, firearms, law, weapons, government, control);
- **cars, with some health** (car, cars, engine, health, water, market): two groups sharing one topic;
- and a **conversational** topic (don, like, just, think, know, people, time, good): not a subject, but a *style*,
  the everyday chat that runs through posts in every group.

There is **no medicine topic**: medical posts are split between the car/health topic and the conversational topic.
LDA found the themes that best explain word co-occurrence, which are not necessarily the newsgroups.

## Step 5: One post, several topics

Look at the mixture of a single post: the main thing that sets topic models apart from clustering.

In [ ]:
TOPIC_NAMES = [f"topic {t}" for t in range(6)]
example = int(np.argmax((mixtures > 0.25).sum(axis=1)))          # a post with at least two sizeable topics
print(f"newsgroup: {group[example]}\n")
print(documents[example][:700].strip(), "...\n")
print(pd.Series(mixtures[example], index=TOPIC_NAMES).round(2).to_string())

**Read the output:** a `sci.med` post about the UK's National Health Service and alternative medicine. Its mixture
is 46% conversational (it is an opinionated reply), 25% the car/health topic (the health service) and 28% the
space topic, whose top words include "research", "center" and "data". That last share is a reminder that a topic
is only a list of words, and a list built mostly from space posts can still serve other subjects. A clustering
would have had to put this post in a single group.

## Step 6: How mixed are posts in general?

In [ ]:
top_share = mixtures.max(axis=1)
print(f"posts whose main topic has more than 80% of the words: {(top_share > 0.8).mean():.0%}")
print(f"posts whose main topic has less than 50%:              {(top_share < 0.5).mean():.0%}")
print("\noverall share of each topic across all posts:")
print(pd.Series(mixtures.mean(axis=0), index=TOPIC_NAMES).round(2).to_string())

**Read the output:** 23% of posts are dominated by one topic (above 80%), while 20% are genuinely mixed, with no
topic above 50%. The conversational topic has by far the largest overall share (37% of all words): everyday chat
is in nearly every post.

## Step 7: Compare with the real newsgroups

Unhide the newsgroups. Cross-tabulate each post's **dominant** topic (its largest share) with its newsgroup, and
compute the ARI from Session 10.

In [ ]:
dominant = mixtures.argmax(axis=1)
print(pd.crosstab(group, dominant, colnames=["dominant topic"]))
print(f"\nARI between dominant topic and newsgroup: {adjusted_rand_score(group, dominant):.3f}")

**Read the output:** hockey (406 posts) and graphics (419) mostly go to "their" topic, and autos mostly to the
car/health topic (315). Space and guns posts are split almost evenly between their own topic and the
conversational one, and most medical posts (426) go to the conversational topic. The ARI is 0.324: clearly
related to the newsgroups, but far from a match.

That is not a failure. LDA was not trying to find the newsgroups: it found the six themes that best explain which
words occur together, and a "how people chat" theme explains a lot of words. When you want to *predict* known
categories, use a classifier (Session 8); topic models are for *exploring* what is in a collection.

## Step 8: How many topics?

Two ways to judge the number of topics:

- **Perplexity on held-out posts:** how surprised the model is by posts it has not seen (lower is better). It
  measures how well the model predicts words, the statistical view.
- **Reading the topics:** can a person name each one? This is the view that matters for exploration.

In [ ]:
test_counts = vectorizer.transform(test_documents)
perplexity = {}
for k in [4, 6, 10, 15]:
    model = LatentDirichletAllocation(n_components=k, learning_method="batch", max_iter=30, random_state=0)
    model.fit(counts)
    perplexity[k] = model.perplexity(test_counts)
    if k == 10:
        lda10 = model
print(pd.Series(perplexity, name="held-out perplexity").round(0).to_string())
print("\nThe 10-topic model:")
show_topics(lda10, words)

**Read the output:** this takes about a minute. Held-out perplexity is lowest at 4 topics (3,117) and rises with
every extra topic (3,792 at 15): by the statistical measure, more topics overfit these few thousand posts.

Yet the **10-topic** model is in some ways *more* useful to read: it finds a clear **medicine** topic (medical,
health, disease, patients, cancer) that the 6-topic model missed. It also splits hockey into the game itself and
league news, and creates a topic of **team abbreviations** (van, det, bos, tor, pit: Vancouver, Detroit, Boston...)
from posts listing scores. Some topics become sharper, some become oddities.

Perplexity and human judgement often disagree; for exploration, people usually trust their reading of the topics,
with perplexity as a sanity check. Try a few values of k and keep the one whose topics are most useful.

## Step 9: NMF, a faster alternative

**Non-negative matrix factorisation (NMF)** approximates the document-term matrix as the product of two smaller
matrices, document × topic and topic × word, with no negative numbers. It has no probabilistic story, but the two
factors are read exactly like LDA's outputs. NMF usually works on **TF-IDF** features and is often faster.

In [ ]:
tfidf = TfidfVectorizer(max_df=0.5, min_df=5, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
weights = tfidf.fit_transform(documents)
nmf = NMF(n_components=6, init="nndsvda", max_iter=500, random_state=0)
nmf_mixtures = nmf.fit_transform(weights)
show_topics(nmf, tfidf.get_feature_names_out())
print(f"\nARI between NMF's dominant topic and newsgroup: {adjusted_rand_score(group, nmf_mixtures.argmax(axis=1)):.3f}")

**Read the output:** NMF runs in a second or two. Several of its topics are crisper than LDA's: **cars** gets its
own topic (car, cars, engine, dealer, price), as do hockey, space and graphics, while guns is mixed with general
discussion. Its ARI with the newsgroups is clearly higher than LDA's (0.474 against 0.324).

One NMF topic is strange: words like "geb", "shameful", "chastity", "skepticism", "intellect", "pitt", "banks". This
is a **signature**: a regular `sci.med` poster ended every post with the same quotation. scikit-learn's footer
removal missed it, and because TF-IDF up-weights rare words that always occur together, NMF gave it a whole topic.
Topic models are very good at finding text that repeats, wanted or not; reading the topics is how you find
cleaning problems.

## Step 10: Topics for new documents

A fitted topic model can describe new text: `transform` gives its topic mixture, using the vocabulary and topics
learned from the training posts.

In [ ]:
new_posts = [
    "The shuttle launch was delayed again; NASA says the orbiter needs new heat tiles before the mission.",
    "My doctor says the new drug lowers blood pressure, but patients report side effects. Any medical advice?",
    "Should the government ban handguns? The law on firearms ownership is too weak, people need control.",
]
new_mixtures = lda10.transform(vectorizer.transform(new_posts))
for post, mixture in zip(new_posts, new_mixtures):
    top = np.argsort(mixture)[::-1][:2]
    print(f"{post[:60]}...  ->  topic {top[0]} ({mixture[top[0]]:.0%}), topic {top[1]} ({mixture[top[1]]:.0%})")

**Read the output:** each new post is assigned mainly to the topic you would expect from Step 8's list: space
(68%), medicine (52%) and guns (81%). The medical question also gets 41% of the conversational topic 7: it is
phrased as a chatty question. Words the vectoriser has never seen are ignored, so a new post must share vocabulary with the
training posts to be described well.

## What this session hands to the module

- **Topic models** describe documents as mixtures of themes, where clustering forces one group per item.
- **LDA** on counts, **NMF** on TF-IDF: read topics by their top words; name them yourself.
- **Choosing the number of topics** is a judgement: perplexity is a guide, readable topics are the goal.
- **Topics reveal the data's quirks:** conversational filler, team abbreviations, repeated signatures.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Giving LDA TF-IDF features | It models counts; results are hard to interpret | `CountVectorizer` for LDA; TF-IDF for NMF |
| Keeping stop words and very common words | Every topic starts with "the", "don", "just" | `stop_words`, `max_df`, and a custom stop list if needed |
| Expecting topics to match known categories | Disappointment; low ARI | Use a classifier for known categories; topic models are for exploration |
| Trusting perplexity alone | Picks models whose topics are hard to use | Read the topics for several k |
| Reading one run as *the* topics | Topics change with the random seed | Fit with two or three seeds; keep themes that recur |
| Skipping text cleaning | Signatures and boilerplate become topics | Inspect the topics; clean and refit |

## Exercises

- [ ] Add the conversational words ("don", "just", "like", "think", "know", "people", "time", "good") to the stop
  words and refit the 6-topic LDA. What replaces the conversational topic?
- [ ] Refit the 6-topic LDA with `random_state=1`. Which topics are the same, which change?
- [ ] Find the `sci.med` signature posts (search the documents for "chastity"), remove that text, and refit NMF.
  What topic appears instead?
- [ ] Use the topic mixtures (`mixtures`) as the input features of a logistic regression that predicts the
  newsgroup. How does its accuracy compare with Session 8's TF-IDF classifier on these six groups?

This is the last session of the module. The [MLOps Skilling Course](../../11.%20MLOps%20Skilling%20Course) picks up
where it ends: tracking, deploying and monitoring models like the ones you have built here.